# 24-Hour Flow Model Training

Runs `experiments/24hr_flow_study.json` through the ML pipeline via `run_experiments.py`.

**Data:** `data/Merged/Miami_GWL_WL_RAIN_GATE_FLOW_2017_2024.csv`  
**Train:** 2017–2023 &nbsp;|&nbsp; **Test:** 2024  
**Target:** `gwl` at +24 h lead time  
**Models:** LR, RF, MLP  
**Results:** `results/<experiment_name>/test/results.csv`

---

## Lag Range Reference

Each input column is represented as a window of lagged (and potentially future) values. The lag range `[a, b]` means features are constructed from hour `a` to hour `b` relative to the prediction time:

- **`[-24, 0]` — historical only:** the model sees observations from 24 h ago up to the current timestep. No future information is provided.
- **`[-24, 24]` — perfect prognostic:** the model also receives values up to 24 h into the future (i.e. out to the full lead time). This simulates a perfect forecast and represents an upper bound on what that variable could contribute if a reliable forecast were available.

---

## Experiment Configurations

### `24hr_flow_all_inputs`
All inputs available at perfect-prog resolution. Establishes the ceiling for flow-augmented performance.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| wl | [-24, 24] | Perfect-prog |
| rain | [-24, 24] | Perfect-prog |
| stgH | [-24, 24] | Perfect-prog |
| stgT | [-24, 24] | Perfect-prog |
| gate1 | [-24, 24] | Perfect-prog |
| gate2 | [-24, 24] | Perfect-prog |
| flow | [-24, 24] | Perfect-prog |

### `24hr_flow_gwl_wl_flow`
Isolates the combined predictive value of water level and flow forecasts, with all other variables restricted to historical observations.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| wl | [-24, 24] | Perfect-prog |
| rain | [-24, 0] | Historical only |
| stgH | [-24, 0] | Historical only |
| stgT | [-24, 0] | Historical only |
| gate1 | [-24, 0] | Historical only |
| gate2 | [-24, 0] | Historical only |
| flow | [-24, 24] | Perfect-prog |

### `24hr_flow_gwl_flow`
Only flow is given perfect-prog knowledge; everything else is historical. Tests whether flow alone can drive 24 h GWL prediction.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| wl | [-24, 0] | Historical only |
| rain | [-24, 0] | Historical only |
| stgH | [-24, 0] | Historical only |
| stgT | [-24, 0] | Historical only |
| gate1 | [-24, 0] | Historical only |
| gate2 | [-24, 0] | Historical only |
| flow | [-24, 24] | Perfect-prog |

### `24hr_flow_gwl_rain_gates`
Flow paired with rain and gate forecasts, but not water level. Tests the flow + forcing variable combination without WL.

| Column | Lag Range | Mode |
|--------|-----------|------|
| gwl | [-24, 0] | Historical only |
| rain | [-24, 24] | Perfect-prog |
| stgH | [-24, 0] | Historical only |
| stgT | [-24, 0] | Historical only |
| gate1 | [-24, 24] | Perfect-prog |
| gate2 | [-24, 24] | Perfect-prog |
| flow | [-24, 24] | Perfect-prog |

Import utilities for JSON parsing, subprocess execution, and path resolution. Resolve the project root and experiment file path relative to this notebook's location in `notebooks/`.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path("../").resolve()  # project root from notebooks/
EXPERIMENT_FILE = ROOT / "experiments" / "24hr_flow_study.json"

## Experiment Configuration

Load the experiment JSON and print each experiment's name, input columns, and model choices — a quick sanity check before kicking off training.

In [2]:
with open(EXPERIMENT_FILE) as f:
    config = json.load(f)

for exp in config["experiments"]:
    print(f"  {exp['experiment_name']}")
    print(f"    inputs: {[s['column'] for s in exp['input_specifications']]}")
    print(f"    models: {exp['model_architectures']}")
    print()

  24hr_flow_all_inputs
    inputs: ['gwl', 'wl', 'rain', 'stgH', 'stgT', 'gate1', 'gate2', 'flow']
    models: ['LR', 'RF', 'MLP']

  24hr_flow_gwl_wl_flow
    inputs: ['gwl', 'wl', 'rain', 'stgH', 'stgT', 'gate1', 'gate2', 'flow']
    models: ['LR', 'RF', 'MLP']

  24hr_flow_gwl_flow
    inputs: ['gwl', 'wl', 'rain', 'stgH', 'stgT', 'gate1', 'gate2', 'flow']
    models: ['LR', 'RF', 'MLP']

  24hr_flow_gwl_rain_gates
    inputs: ['gwl', 'rain', 'stgH', 'stgT', 'gate1', 'gate2', 'flow']
    models: ['LR', 'RF', 'MLP']



## Run Training

Launch `run_experiments.py` as a subprocess with `cwd` set to the project root so all its internal relative paths resolve correctly. Training output streams live to this cell. Raises an error if the script exits with a non-zero code.

In [3]:
result = subprocess.run(
    [sys.executable, "run_experiments.py", "-e", "experiments/24hr_flow_study.json"],
    cwd=ROOT,
    capture_output=False,
)

if result.returncode != 0:
    raise RuntimeError(f"run_experiments.py exited with code {result.returncode}")

Test results exist for model LR for experiment 24hr_flow_all_inputs
Test results exist for model RF for experiment 24hr_flow_all_inputs
Test results exist for model MLP for experiment 24hr_flow_all_inputs
Test results exist for model LR for experiment 24hr_flow_gwl_wl_flow
Test results exist for model RF for experiment 24hr_flow_gwl_wl_flow
Test results exist for model MLP for experiment 24hr_flow_gwl_wl_flow
Test results exist for model LR for experiment 24hr_flow_gwl_flow
Test results exist for model RF for experiment 24hr_flow_gwl_flow
Test results exist for model MLP for experiment 24hr_flow_gwl_flow
Test results exist for model LR for experiment 24hr_flow_gwl_rain_gates
Test results exist for model RF for experiment 24hr_flow_gwl_rain_gates
Test results exist for model MLP for experiment 24hr_flow_gwl_rain_gates


## Results

Read each experiment's `test/results.csv`, prepend the experiment name as a column, and concatenate into a single summary DataFrame for comparison across all four experiments and three models.

In [4]:
import pandas as pd

rows = []
for exp in config["experiments"]:
    path = ROOT / "results" / exp["experiment_name"] / "test" / "results.csv"
    if path.exists():
        df = pd.read_csv(path, index_col=0)
        df.insert(0, "experiment", exp["experiment_name"])
        rows.append(df)
    else:
        print(f"WARNING: not found — {path}")

pd.concat(rows, ignore_index=True)

,experiment,CF_15CM,CF_5CM,CF_1CM,MSE,RMSE,MAE,MEDAE,MAPE,R2,model
0,24hr_flow_all_inputs,99.914809,97.075110,67.883004,0.000349,0.018680,0.010525,0.006112,1.617890e+10,0.974525,LR
1,24hr_flow_all_inputs,99.077098,90.983956,57.901462,0.001234,0.035124,0.018121,0.007908,1.613770e+11,0.909928,RF
2,24hr_flow_all_inputs,96.592361,80.604856,48.913815,0.003592,0.059934,0.031169,0.010642,6.407598e+10,0.737747,MLP
3,24hr_flow_gwl_wl_flow,98.920914,89.209144,44.015334,0.002130,0.046152,0.022898,0.011587,1.447965e+11,0.844488,LR
4,24hr_flow_gwl_wl_flow,98.906716,89.734488,53.656112,0.001483,0.038512,0.019920,0.008950,1.812214e+11,0.891718,RF
5,24hr_flow_gwl_wl_flow,49.822519,44.029533,32.131194,0.054092,0.232576,0.158904,0.152400,1.110943e+11,-2.949177,MLP
6,24hr_flow_gwl_flow,98.949311,88.854181,47.564958,0.002145,0.046311,0.022749,0.010830,1.377395e+11,0.843415,LR
7,24hr_flow_gwl_flow,98.892517,89.919069,53.514128,0.001406,0.037503,0.019832,0.008958,1.789873e+11,0.897315,RF
8,24hr_flow_gwl_flow,95.541673,77.254011,45.747551,0.007147,0.084543,0.039110,0.012246,1.950018e+11,0.478169,MLP
9,24hr_flow_gwl_rain_gates,99.204884,91.665483,58.398410,0.001376,0.037100,0.017879,0.007796,8.137833e+10,0.899508,LR
